In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.retail_training.data_quality_results (
    pipeline_name STRING,
    check_name STRING,
    issue_count BIGINT,
    status STRING,
    checked_at TIMESTAMP
)
USING DELTA;


In [0]:

import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    LongType, TimestampType
)

# Task audit configuration
PIPELINE_NAME = "retail_data_pipeline"
TASK_NAME = "06_data_quality_checks"
AUDIT_TABLE = "workspace.retail_training.pipeline_audit_log"

RUN_ID = str(uuid.uuid4())
START_TIME = datetime.now(timezone.utc)
ROWS_PROCESSED = 0

print(f"Task audit ID: {RUN_ID}")


def write_audit_record(
    run_status,
    start_time,
    end_time,
    rows_processed=None,
    error_message=None
):
    audit_schema = StructType([
        StructField("pipeline_name", StringType(), False),
        StructField("task_name", StringType(), False),
        StructField("run_id", StringType(), True),
        StructField("run_status", StringType(), False),
        StructField("start_time", TimestampType(), True),
        StructField("end_time", TimestampType(), True),
        StructField("rows_processed", LongType(), True),
        StructField("error_message", StringType(), True),
        StructField("logged_at", TimestampType(), True)
    ])

    audit_record = [(
        PIPELINE_NAME,
        TASK_NAME,
        RUN_ID,
        run_status,
        start_time,
        end_time,
        rows_processed,
        error_message[:4000] if error_message else None,
        end_time
    )]

    audit_df = spark.createDataFrame(
        audit_record,
        schema=audit_schema
    )

    audit_df.write.format("delta").mode("append").saveAsTable(
        AUDIT_TABLE
    )


try:
    checked_at = datetime.now(timezone.utc)

    # Load Silver sales and Bronze dimensions
    sales = spark.table("workspace.retail_training.silver_sales")
    customers = spark.table("workspace.retail_training.bronze_customers")
    products = spark.table("workspace.retail_training.bronze_products")
    stores = spark.table("workspace.retail_training.bronze_stores")

    # Count invalid records
    duplicate_ids = (
        sales.groupBy("sale_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    null_sale_ids = sales.filter(
        F.col("sale_id").isNull()
    ).count()

    invalid_quantity = sales.filter(
        F.col("quantity").isNull() | (F.col("quantity") <= 0)
    ).count()

    invalid_price = sales.filter(
        F.col("unit_price").isNull() | (F.col("unit_price") < 0)
    ).count()

    missing_customers = (
        sales.join(
            customers.select("customer_id").distinct(),
            on="customer_id",
            how="left_anti"
        ).count()
    )

    missing_products = (
        sales.join(
            products.select("product_id").distinct(),
            on="product_id",
            how="left_anti"
        ).count()
    )

    missing_stores = (
        sales.join(
            stores.select("store_id").distinct(),
            on="store_id",
            how="left_anti"
        ).count()
    )

    # Build the results
    results = [
        ("Duplicate sale IDs", duplicate_ids),
        ("Null sale IDs", null_sale_ids),
        ("Invalid quantities", invalid_quantity),
        ("Invalid prices", invalid_price),
        ("Missing customers", missing_customers),
        ("Missing products", missing_products),
        ("Missing stores", missing_stores),
    ]

    quality_rows = [
        (
            PIPELINE_NAME,
            check_name,
            int(issue_count),
            "PASS" if issue_count == 0 else "FAIL",
            checked_at
        )
        for check_name, issue_count in results
    ]

    quality_schema = StructType([
        StructField("pipeline_name", StringType(), False),
        StructField("check_name", StringType(), False),
        StructField("issue_count", LongType(), False),
        StructField("status", StringType(), False),
        StructField("checked_at", TimestampType(), False)
    ])

    quality_df = spark.createDataFrame(
        quality_rows,
        schema=quality_schema
    )

    # Save detailed results before raising any quality exception
    quality_df.write.format("delta").mode("append").saveAsTable(
        "workspace.retail_training.data_quality_results"
    )

    display(quality_df)

    failed_count = sum(
        1 for _, issue_count in results
        if issue_count != 0
    )

    print(f"Total checks: {len(results)}")
    print(f"Passed: {len(results) - failed_count}")
    print(f"Failed: {failed_count}")

    if failed_count > 0:
        raise RuntimeError(
            f"DATA QUALITY FAILURE: {failed_count} check(s) found issues. "
            "Review data_quality_results."
        )

    ROWS_PROCESSED = len(results)
    print("SUCCESS: All detailed data quality checks passed.")

except Exception as e:
    end_time = datetime.now(timezone.utc)

    try:
        write_audit_record(
            run_status="FAILED",
            start_time=START_TIME,
            end_time=end_time,
            rows_processed=ROWS_PROCESSED,
            error_message=str(e)
        )
        print("FAILED audit record written.")
    except Exception as audit_error:
        print(f"WARNING: Could not write failure audit record: {audit_error}")

    raise

else:
    end_time = datetime.now(timezone.utc)

    write_audit_record(
        run_status="SUCCESS",
        start_time=START_TIME,
        end_time=end_time,
        rows_processed=ROWS_PROCESSED,
        error_message=None
    )

    print(f"SUCCESS audit record written: {ROWS_PROCESSED} checks completed.")


Task audit ID: 0055e0f1-c6fd-47b0-9e7f-03ba43c3952a


pipeline_name,check_name,issue_count,status,checked_at
retail_data_pipeline,Duplicate sale IDs,0,PASS,2026-10-20T06:29:11.593Z
retail_data_pipeline,Null sale IDs,0,PASS,2026-10-20T06:29:11.593Z
retail_data_pipeline,Invalid quantities,0,PASS,2026-10-20T06:29:11.593Z
retail_data_pipeline,Invalid prices,0,PASS,2026-10-20T06:29:11.593Z
retail_data_pipeline,Missing customers,0,PASS,2026-10-20T06:29:11.593Z
retail_data_pipeline,Missing products,0,PASS,2026-10-20T06:29:11.593Z
retail_data_pipeline,Missing stores,0,PASS,2026-10-20T06:29:11.593Z


Total checks: 7
Passed: 7
Failed: 0
SUCCESS: All detailed data quality checks passed.
SUCCESS audit record written: 7 checks completed.
